# 16c — Counterfactual Search

Loads pre-mined constraints (from 16a) and pre-trained VAE (from 16b),
then runs REVISED+ counterfactual search. No mining or training happens here.

In [1]:
import sys
import os
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

# src/ must be on sys.path for torch.load to unpickle event_log_loader classes
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [2]:
# ===== TEST MODE =====
TEST_MODE = False
TEST_N_SEQUENCES = 50

In [ ]:
import torch

# --- Load dataset + prediction model ---
data_path = _current / 'encoded_data' / 'BPIC_2017_all_5_test.pkl'
full_dataset = torch.load(data_path, weights_only=False)

if TEST_MODE:
    dataset = [full_dataset[i] for i in range(min(TEST_N_SEQUENCES, len(full_dataset)))]
    print(f"TEST MODE: Using {len(dataset)} sequences (subset of {len(full_dataset)})")
else:
    dataset = full_dataset

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f"Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}")

from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

model_path = _current / 'src' / 'notebooks' / 'training_variational_dropout' / 'BPIC17' / 'BPIC_2017_full_grad_norm_new_4layer.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(model_path), dropout=0.0)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

# --- TensorDecoder + activity vocabulary ---
from src.interpretability.utils.tensor_decoder import TensorDecoder

decoder = TensorDecoder(full_dataset)

ACTIVITY_FEATURE = 'concept:name'
activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]
eos_idx = next(i for i, name in enumerate(activity_names) if name == 'EOS')

print(f"Activity vocabulary ({len(activity_names)}), EOS={eos_idx}")

Dataset: 253196 sequences, 9 cat, 9 num, seq_len=96


In [ ]:
# Load pre-mined constraints + pre-trained VAE
import pickle

constraints_pkl_name = 'bpic17_constraints_test.pkl' if TEST_MODE else 'bpic17_constraints.pkl'
constraints_pkl_path = _current / 'encoded_data' / constraints_pkl_name
with open(constraints_pkl_path, 'rb') as f:
    constraints_data = pickle.load(f)

all_constraints = constraints_data['all']
data_conditions = constraints_data['data_conditions']
print(f"Constraints: {len(all_constraints)} (from {constraints_pkl_path.name})")

vae_pkl_name = 'bpic17_vae_test.pkl' if TEST_MODE else 'bpic17_vae.pkl'
vae_path = str(_current / 'encoded_data' / vae_pkl_name)
print(f"VAE path: {vae_pkl_name}")

In [ ]:
from src.interpretability.perturbation_methods import RevisedPlus, RevisedPlusConfig, create_revised_plus_for_model

device = 'mps' if torch.backends.mps.is_available() else 'cpu'

config = RevisedPlusConfig(
    vae_epochs=300 if TEST_MODE else 100,
    vae_kl_weight=0.1,
    declare_min_support=0.9,
    n_candidates_per_round=200,
    n_search_rounds=5,
    top_k=5,
    min_plausibility=0.0,
    device=device,
    activity_feature=ACTIVITY_FEATURE,
)

# Create RevisedPlus with pre-mined constraints and pre-trained VAE
# No mining, no training — just loads both
rp = create_revised_plus_for_model(
    model=model,
    dataset=dataset,
    activity_names=activity_names,
    config=config,
    vae_path=vae_path,
    constraints=all_constraints,
    data_conditions=data_conditions,
)
print(f"\nREVISED+ ready (no mining, no training):")
print(f"  VAE parameters: {sum(p.numel() for p in rp.vae.parameters()):,}")
print(f"  All constraints: {len(rp.all_constraints)}")
print(f"  Prefix-safe constraints: {len(rp.prefix_safe_constraints)}")

In [ ]:
import numpy as np

# Find a good candidate: in-progress prefix with uncertain prediction
best_idx, best_prob = None, 1.0
search_limit = min(50 if TEST_MODE else 200, len(dataset))
for i in range(search_limit):
    cat_t, num_t, _ = dataset[i]
    act = cat_t[0]
    if (act == eos_idx).any():
        continue
    cat_in = [c.unsqueeze(0) for c in cat_t]
    num_in = [n.unsqueeze(0) for n in num_t]
    with torch.no_grad():
        preds = model((cat_in, num_in))[0]
        logits = preds[0][f"{ACTIVITY_FEATURE}_mean"][0]
        p = torch.softmax(logits, dim=-1)
        top_p = p.max().item()
    if 0.4 < top_p < best_prob:
        best_idx, best_prob = i, top_p

test_idx = best_idx if best_idx is not None else 42
print(f"Selected test_idx={test_idx} (top_p={best_prob:.3f})")

cat_tuple, num_tuple, case_id = dataset[test_idx]

print(f"\nCase: {case_id}")
df_orig = decoder.decode_sequence(cat_tuple, num_tuple, case_id=case_id)
display(df_orig)

In [ ]:
cat_tensors = [c.unsqueeze(0) for c in cat_tuple]
num_tensors = [n.unsqueeze(0) for n in num_tuple]

explanation = rp.explain(cat_tensors, num_tensors, target_class=None)
print(explanation)

In [ ]:
if explanation.counterfactuals:
    best = explanation.get_best()

    print("=" * 80)
    print("ORIGINAL PREFIX")
    print(f"Prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})")
    print(f"Prefix length: {explanation.prefix_len} events")
    print("=" * 80)
    display(df_orig)

    print(f"\n{'=' * 80}")
    print("BEST COUNTERFACTUAL PREFIX")
    print(f"Prediction: {best.counterfactual_prediction_name} (p={best.counterfactual_probability:.3f})")
    cf_prefix_len = len(best.activity_sequence)
    print(f"Prefix length: {cf_prefix_len} events (delta={cf_prefix_len - explanation.prefix_len:+d})")
    print(f"Proximity: {best.proximity:.3f}  Sparsity: {best.sparsity}")
    print(f"Feasibility: {best.feasibility:.3f}")
    print(f"Plausibility (definite): {best.plausibility_definite:.2f}")
    print(f"Plausibility (optimistic): {best.plausibility_optimistic:.2f}")
    print(f"Combined score: {best.combined_score:.4f}")
    print("=" * 80)

    cf_cat = tuple(best.cat_sequence)
    cf_num = tuple(best.num_sequence[:, i] for i in range(best.num_sequence.shape[1])) if best.num_sequence is not None else num_tuple
    df_cf = decoder.decode_sequence(cf_cat, cf_num, skip_padding=False)
    display(df_cf)
else:
    print("No counterfactuals found. Try increasing n_search_rounds or noise_scale.")

In [ ]:
import pandas as pd

if explanation.counterfactuals:
    rows = []
    for i, cf in enumerate(explanation.counterfactuals):
        rows.append({
            'rank': i + 1,
            'prediction': cf.counterfactual_prediction_name,
            'probability': f"{cf.counterfactual_probability:.3f}",
            'prefix_len': len(cf.activity_sequence),
            'activities': ' -> '.join(activity_names[a] for a in cf.activity_sequence),
            'proximity': f"{cf.proximity:.2f}",
            'sparsity': cf.sparsity,
            'feasibility': f"{cf.feasibility:.3f}",
            'plaus_def': f"{cf.plausibility_definite:.2f}",
            'plaus_opt': f"{cf.plausibility_optimistic:.2f}",
            'score': f"{cf.combined_score:.4f}",
        })

    df_cfs = pd.DataFrame(rows)
    print(f"Original: {' -> '.join(activity_names[a] for a in explanation.original_activity_sequence)}")
    print(f"Original prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})")
    print()
    display(df_cfs)